### **Data Pre-Processing (Part 1)**

***ASSIGNMENT 3***

***DATA CLEANING ,DATA TRANSFORMATION and DATA INTEGRATION***

#### **Import Libries**

In [ ]:
import glob
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

#### **Load files**

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Group's Google Drive
    DATA_FOLDER = "/content/drive/MyDrive/COMP541 - Group 1/Prepped Data Files (for merging)/alternate integration output"
else:
    # Local / non-Colab fallback: look in the same folder as this script, then ../data/cleaned.
    DATA_FOLDER = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
    for cand in [DATA_FOLDER,
                 os.path.join(DATA_FOLDER, "..", "data", "cleaned"),
                 os.path.join(DATA_FOLDER, "data", "cleaned")]:
        if os.path.exists(os.path.join(cand, "kaggle_cost_of_living_prepared.csv")):
            DATA_FOLDER = os.path.abspath(cand)
            break

OUTPUT_DIR = os.path.join(DATA_FOLDER, "alternate integration output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

INCLUDE_TRIPADVISOR = True
print("DATA_FOLDER:", DATA_FOLDER)
print("OUTPUT_DIR :", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_FOLDER: /content/drive/MyDrive/COMP541 - Group 1/Prepped Data Files (for merging)
OUTPUT_DIR : /content/drive/MyDrive/COMP541 - Group 1/Prepped Data Files (for merging)/alternate integration output


#### **File discovery + safe CSV reader**

In [ ]:
FILE_KEYWORDS = {
    "flight": ["flight"],
    "airbnb": ["airbnb"],
    "col": ["cost_of_living", "cost of living"],
    "tripadvisor": ["tripadvisor"],
    "weather": ["weather"],
    "worldbank": ["worldbank", "world bank"],
}

In [ ]:
def find_file(keywords):
    for fname in os.listdir(DATA_FOLDER):
        lower = fname.lower()
        if any(k in lower for k in keywords) and lower.endswith((".csv", ".xlsx", ".xls")):
            return fname
    raise FileNotFoundError(
        f"Could not find a file matching {keywords} in {DATA_FOLDER}. "
        f"Files present: {os.listdir(DATA_FOLDER)}"
    )


def read_csv_safely(file_path, usecols=None, on_bad_lines="error"):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1", "ISO-8859-1"]:
        try:
            return pd.read_csv(file_path, encoding=enc, usecols=usecols, on_bad_lines=on_bad_lines)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(file_path, encoding="cp1252", encoding_errors="replace",
                       usecols=usecols, on_bad_lines=on_bad_lines)


def get_path(role):
    return os.path.join(DATA_FOLDER, find_file(FILE_KEYWORDS[role]))


def section(title):
    print("\n" + "=" * 78 + "\n" + title + "\n" + "=" * 78)


def sub(title):
    print("\n--- " + title + " ---")

#### **Hepler Functions**

In [ ]:
def strip_artifact_characters(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = re.sub(r"[\"\'{}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# Manual alias tables built by comparing the actual city/country spellings
COUNTRY_ALIASES = {
    "the netherlands": "netherlands",
    "hong kong sar china": "hong kong",
    "viet nam": "vietnam",
    "china hong kong": "hong kong",
}
CITY_ALIASES = {"new york": "new york city"}

In [ ]:
# Turns a messy text field into a clean join key.
def make_key(text, alias_table=None):
    if pd.isna(text):
        return np.nan
    text = strip_artifact_characters(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if alias_table and text in alias_table:
        text = alias_table[text]
    return text

In [ ]:
def cap_outliers_iqr(df, cols, factor=1.5):
    capped = 0
    for c in cols:
        if c not in df.columns:
            continue
        q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - factor * iqr, q3 + factor * iqr
        capped += int(((df[c] < low) | (df[c] > high)).sum())
        df[c] = df[c].clip(lower=low, upper=high)
    return capped

In [ ]:
def min_max_scale(df, cols, suffix="_minmax"):
    for c in cols:
        if c not in df.columns:
            continue
        col = df[c].astype(float)
        rng = col.max() - col.min()
        df[c + suffix] = 0.0 if rng == 0 else (col - col.min()) / rng
    return df

In [ ]:
def z_score_scale(df, cols, suffix="_zscore"):
    for c in cols:
        if c not in df.columns:
            continue
        col = df[c].astype(float)
        std = col.std()
        df[c + suffix] = 0.0 if std == 0 else (col - col.mean()) / std
    return df

####  **Load,Clean and pre-aggregated raw files**

In [ ]:
NUMBEO_ITEM_MAP = {
    "x1": "cost_meal_inexpensive_restaurant", "x2": "cost_meal_for_two_midrange_restaurant",
    "x3": "cost_fastfood_combo_meal", "x4": "cost_domestic_draft_beer", "x5": "cost_imported_beer",
    "x8": "cost_milk_1l", "x9": "cost_white_bread_500g", "x10": "cost_white_rice_1kg_UNVERIFIED",
    "x18": "cost_oneway_transport_ticket", "x19": "cost_chicken_fillets_1kg",
    "x20": "cost_monthly_transport_pass", "x52": "price_per_sqm_apartment_city_centre",
    "x53": "price_per_sqm_apartment_outside_centre",
}
DINING_COLS = ["cost_meal_inexpensive_restaurant", "cost_meal_for_two_midrange_restaurant",
               "cost_fastfood_combo_meal", "cost_domestic_draft_beer", "cost_imported_beer"]
GROCERY_COLS = ["cost_milk_1l", "cost_white_bread_500g", "cost_white_rice_1kg_UNVERIFIED",
                "cost_chicken_fillets_1kg"]
TRANSPORT_COLS = ["cost_oneway_transport_ticket", "cost_monthly_transport_pass"]
HOUSING_COLS = ["price_per_sqm_apartment_city_centre", "price_per_sqm_apartment_outside_centre"]


def load_clean_cost_of_living():
    sub("Kaggle cost-of-living index")
    df = read_csv_safely(get_path("col"))
    before = df.shape
    df = df.drop_duplicates(subset=["destination_key"])
    for c in ["city", "country"]:
        df[c] = df[c].astype(str).str.strip()
    cap_outliers_iqr(df, [c for c in df.columns if c.startswith("x")])
    rename_map = {k: v for k, v in NUMBEO_ITEM_MAP.items() if k in df.columns}
    df = df.rename(columns=rename_map)
    df["dining_out_cost_index"] = df[DINING_COLS].mean(axis=1)
    df["groceries_cost_index"] = df[GROCERY_COLS].mean(axis=1)
    df["transport_cost_index"] = df[TRANSPORT_COLS].mean(axis=1)
    df["housing_cost_index"] = df[HOUSING_COLS].mean(axis=1)
    df["cost_of_living_composite"] = df[DINING_COLS + GROCERY_COLS + TRANSPORT_COLS + HOUSING_COLS].mean(axis=1)
    df["city_key"] = df["city"].apply(make_key, alias_table=CITY_ALIASES)
    df["country_key"] = df["country"].apply(make_key, alias_table=COUNTRY_ALIASES)
    df = df[~df.duplicated(subset=["city_key", "country_key"], keep="first")]
    keep = (["city", "country", "city_key", "country_key", "cost_of_living_composite",
             "dining_out_cost_index", "groceries_cost_index", "transport_cost_index",
             "housing_cost_index"] + list(rename_map.values()))
    print(f"cost_of_living: {before} -> {df.shape}")
    return df[keep]


def load_clean_worldbank():
    sub("World Bank tourism statistics")
    df = read_csv_safely(get_path("worldbank"))
    before = df.shape
    df = df.drop_duplicates(subset=["country_code"])
    df["country_name"] = df["country_name"].astype(str).str.strip()
    cap_outliers_iqr(df, ["tourism_arrivals_count", "tourism_receipts_usd",
                          "tourism_receipts_per_arrival_usd"])
    df["country_key"] = df["country_name"].apply(make_key, alias_table=COUNTRY_ALIASES)
    print(f"worldbank: {before} -> {df.shape}")
    return df[["country_name", "country_key", "tourism_arrivals_count",
               "tourism_receipts_usd", "tourism_receipts_per_arrival_usd"]]


def load_clean_weather():
    sub("Monthly weather/climate data")
    df = read_csv_safely(get_path("weather"))
    before = df.shape
    df = df.drop_duplicates()
    df["city"] = df["city"].astype(str).str.strip()
    cap_outliers_iqr(df, ["temp_c", "precipitation", "humidity", "clouds", "wind",
                          "pct_pleasant", "pct_dry", "pct_sunny", "weather_score"])
    df["city_key"] = df["city"].apply(make_key, alias_table=CITY_ALIASES)
    annual = df.groupby("city_key").agg(
        avg_temp_c=("temp_c", "mean"), avg_precipitation=("precipitation", "mean"),
        avg_humidity=("humidity", "mean"), avg_pct_pleasant_days=("pct_pleasant", "mean"),
        avg_pct_sunny_days=("pct_sunny", "mean"), avg_weather_score=("weather_score", "mean"),
    ).reset_index()
    print(f"weather: {before} -> aggregated to {annual.shape}")
    return annual


def load_clean_flight():
    sub("Flight price data (LAX outbound)")
    df = read_csv_safely(get_path("flight"))
    before = df.shape
    df = df.drop_duplicates()
    df = df.drop(columns=[c for c in df.columns if c.endswith(("_mm", "_z", "_Imputed"))])
    for c in ["Dest_City", "Dest_Country", "Dest_Region", "Carrier"]:
        df[c] = df[c].astype(str).str.strip()
    cap_outliers_iqr(df, ["Flight_Price_USD", "Duration_Min", "Price_Per_Hour"])
    df["city_key"] = df["Dest_City"].apply(make_key, alias_table=CITY_ALIASES)
    df["country_key"] = df["Dest_Country"].apply(make_key, alias_table=COUNTRY_ALIASES)
    agg = df.groupby(["city_key", "country_key"]).agg(
        city=("Dest_City", "first"), country=("Dest_Country", "first"),
        flight_price_avg_usd=("Flight_Price_USD", "mean"),
        flight_price_min_usd=("Flight_Price_USD", "min"),
        flight_duration_avg_min=("Duration_Min", "mean"),
        flight_avg_stops=("Stops", "mean"),
        flight_pct_nonstop=("Is_Nonstop", "mean"),
        flight_quotes_found=("Flight_Price_USD", "size"),
    ).reset_index()
    agg["flight_pct_nonstop"] = (agg["flight_pct_nonstop"] * 100).round(1)
    print(f"flight: {before} -> aggregated to {agg.shape}")
    return agg


def load_clean_airbnb():
    sub("Inside Airbnb destination summary")
    df = read_csv_safely(get_path("airbnb"))
    before = df.shape
    df = df.drop_duplicates().dropna(subset=["usd_rate"])
    for c in ["destination", "country", "region"]:
        df[c] = df[c].astype(str).str.strip()
    df["region"] = df["region"].replace("nan", np.nan)
    cap_outliers_iqr(df, ["avg_airbnb_price_usd", "median_airbnb_price_usd",
                          "min_airbnb_price_usd", "max_airbnb_price_usd",
                          "estimated_7_day_lodging_cost_usd"])
    df = df.rename(columns={"destination": "city", "region": "airbnb_subregion"})
    df["city_key"] = df["city"].apply(make_key, alias_table=CITY_ALIASES)
    df["country_key"] = df["country"].apply(make_key, alias_table=COUNTRY_ALIASES)
    keep = ["city", "country", "city_key", "country_key", "airbnb_subregion",
            "avg_airbnb_price_usd", "median_airbnb_price_usd", "min_airbnb_price_usd",
            "estimated_7_day_lodging_cost_usd"]
    print(f"airbnb: {before} -> {df.shape}")
    return df[keep]


def load_clean_tripadvisor():
    sub("TripAdvisor hotel reviews (large file)")
    usecols = ["hotel_class", "City", "hotel_id", "Service", "Cleanliness", "Overall",
               "Value", "Location", "Sleep_quality", "Room"]
    df = pd.read_csv(get_path("tripadvisor"), encoding="cp1252", encoding_errors="replace",
                     usecols=usecols, on_bad_lines="skip", low_memory=False)
    df["City"] = df["City"].apply(strip_artifact_characters)
    df = df[~(df["City"].isin(["locality", "nan", ""]) | df["City"].isna())]
    rating_cols = ["Service", "Cleanliness", "Overall", "Value", "Location", "Sleep_quality", "Room"]
    for c in rating_cols + ["hotel_class"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in rating_cols:
        df[c] = df[c].clip(1, 5)
    df["city_key"] = df["City"].apply(make_key, alias_table=CITY_ALIASES)
    summary = df.groupby("city_key").agg(
        hotel_review_count=("Overall", "size"), unique_hotels=("hotel_id", "nunique"),
        avg_hotel_class=("hotel_class", "mean"), avg_overall_rating=("Overall", "mean"),
        avg_value_rating=("Value", "mean"), avg_location_rating=("Location", "mean"),
    ).reset_index()
    print(f"tripadvisor: aggregated to {summary.shape}")
    return summary

#### **Data Integration - Build the destination list, then merge**

In [ ]:
def build_union_spine(col, flight, airbnb):
    sub("Building the union destination list (cost of living + flights + Airbnb)")
    parts = [col[["city_key", "country_key"]], flight[["city_key", "country_key"]],
             airbnb[["city_key", "country_key"]]]
    spine = pd.concat(parts, ignore_index=True).dropna(subset=["city_key"]).drop_duplicates()

    # Readable city/country labels, preferring cost-of-living, then flights, then Airbnb
    city_name, country_name = {}, {}
    for src in [airbnb, flight, col]:  # last wins -> col preferred
        for ck, name in zip(src["city_key"], src["city"]):
            if pd.notna(ck):
                city_name[ck] = name
        for nk, name in zip(src["country_key"], src["country"]):
            if pd.notna(nk):
                country_name[nk] = name
    spine["city"] = spine["city_key"].map(city_name)
    spine["country"] = spine["country_key"].map(country_name)
    print(f"union destination list: {len(spine)} unique destinations")
    print(f"  (cost-of-living: {col[['city_key','country_key']].drop_duplicates().shape[0]}, "
          f"flights: {flight.shape[0]}, airbnb: {airbnb[['city_key','country_key']].drop_duplicates().shape[0]})")
    return spine


def integrate_union(spine, col, airbnb, weather, flight, worldbank, tripadvisor=None):
    sub("Merging all sources onto the UNION destination list")
    base = spine.copy()
    n = len(base)

    base = base.merge(col.drop(columns=["city", "country"]), on=["city_key", "country_key"], how="left")
    print(f"+ cost of living matched: {base['cost_of_living_composite'].notna().sum()}/{n}")

    base = base.merge(airbnb.drop(columns=["city", "country"]), on=["city_key", "country_key"], how="left")
    print(f"+ airbnb matched:         {base['avg_airbnb_price_usd'].notna().sum()}/{n}")

    base = base.merge(flight.drop(columns=["city", "country"]), on=["city_key", "country_key"], how="left")
    print(f"+ flight price matched:   {base['flight_price_avg_usd'].notna().sum()}/{n}")

    base = base.merge(weather, on="city_key", how="left")
    print(f"+ weather matched:        {base['avg_weather_score'].notna().sum()}/{n}")

    base = base.merge(worldbank.drop(columns=["country_name"]), on="country_key", how="left")
    print(f"+ tourism matched:        {base['tourism_arrivals_count'].notna().sum()}/{n}")

    if tripadvisor is not None:
        base = base.merge(tripadvisor, on="city_key", how="left")
        print(f"+ hotel reviews matched:  {base['avg_overall_rating'].notna().sum()}/{n}")
    return base

#### **Post-merge: coverage flags + transforms**

In [ ]:
def flag_and_finish(df):
    section("POST-MERGE COVERAGE FLAGS")
    probes = {
        "has_cost_of_living": "cost_of_living_composite",
        "has_airbnb_data": "avg_airbnb_price_usd",
        "has_flight_data": "flight_price_avg_usd",
        "has_weather_data": "avg_weather_score",
        "has_tourism_stats": "tourism_arrivals_count",
    }
    if "avg_overall_rating" in df.columns:
        probes["has_hotel_review_data"] = "avg_overall_rating"
    for flag, probe in probes.items():
        df[flag] = df[probe].notna().astype(int)

    if "airbnb_subregion" in df.columns:
        df["airbnb_subregion"] = df["airbnb_subregion"].fillna("Unknown")

    scale_cols = ["cost_of_living_composite", "avg_weather_score",
                  "avg_airbnb_price_usd", "flight_price_avg_usd"]
    df = min_max_scale(df, scale_cols)
    df = z_score_scale(df, scale_cols)

    rank_pct = df["cost_of_living_composite"].rank(pct=True)
    df["cost_of_living_tier"] = pd.cut(rank_pct, bins=[-0.01, 1/3, 2/3, 1.0],
                                       labels=["Low", "Medium", "High"])
    df["flight_price_tier"] = pd.cut(df["flight_price_avg_usd"],
                                     bins=[-np.inf, 400, 800, np.inf],
                                     labels=["Budget", "Mid-range", "Premium"])
    df["destination_value_score"] = (
        (1 - df["cost_of_living_composite_minmax"]) * (1/3)
        + df["avg_weather_score_minmax"] * (1/3)
        + (1 - df["avg_airbnb_price_usd_minmax"]) * (1/3)
    ).round(3)
    return df

#### **Main Pipeline**

In [ ]:
col = load_clean_cost_of_living()
airbnb = load_clean_airbnb()
weather = load_clean_weather()
flight = load_clean_flight()
worldbank = load_clean_worldbank()
tripadvisor = load_clean_tripadvisor() if INCLUDE_TRIPADVISOR else None

spine = build_union_spine(col, flight, airbnb)
merged = integrate_union(spine, col, airbnb, weather, flight, worldbank, tripadvisor)
merged = flag_and_finish(merged)

full_path = os.path.join(OUTPUT_DIR, "integrated_destinations_union.csv")
merged.to_csv(full_path, index=False)

candidates = merged[merged["has_flight_data"] == 1].copy()
cand_path = os.path.join(OUTPUT_DIR, "modeling_candidates_union.csv")
candidates.to_csv(cand_path, index=False)

section("FINAL OUTPUT")
print(f"Full integrated table:     {merged.shape[0]} rows x {merged.shape[1]} cols -> {full_path}")
print(f"Modeling-candidate subset: {candidates.shape[0]} rows x {candidates.shape[1]} cols -> {cand_path}")


--- Kaggle cost-of-living index ---
cost_of_living: (4956, 16) -> (4956, 23)

--- Inside Airbnb destination summary ---
airbnb: (103, 22) -> (102, 24)

--- Monthly weather/climate data ---
weather: (1608, 14) -> aggregated to (134, 7)

--- Flight price data (LAX outbound) ---
flight: (98, 22) -> aggregated to (49, 10)

--- World Bank tourism statistics ---
worldbank: (198, 11) -> (198, 12)

--- TripAdvisor hotel reviews (large file) ---
tripadvisor: aggregated to (25, 7)

--- Building the union destination list (cost of living + flights + Airbnb) ---
union destination list: 4999 unique destinations
  (cost-of-living: 4956, flights: 49, airbnb: 102)

--- Merging all sources onto the UNION destination list ---
+ cost of living matched: 4956/4999
+ airbnb matched:         102/4999
+ flight price matched:   49/4999
+ weather matched:        144/4999
+ tourism matched:        4775/4999
+ hotel reviews matched:  25/4999

POST-MERGE COVERAGE FLAGS

FINAL OUTPUT
Full integrated table:     499

In [ ]:
section("COVERAGE: union list (this notebook) vs current Airbnb-based merge")
print(f"{'metric':<26}{'Airbnb (current)':>20}{'Union (proposed)':>20}")
rows = [
    ("base size",            "102", f"{len(merged):,}"),
    ("flight destinations",  "21",  f"{int(merged['has_flight_data'].sum())}"),
    ("cost of living",       "64",  f"{int(merged['has_cost_of_living'].sum())}"),
    ("airbnb lodging",       "102", f"{int(merged['has_airbnb_data'].sum())}"),
    ("weather",              "95",  f"{int(merged['has_weather_data'].sum())}"),
    ("tourism",              "101", f"{int(merged['has_tourism_stats'].sum())}"),
]
for name, a, u in rows:
    print(f"{name:<26}{a:>20}{u:>20}")

print(f"\nModeling candidates (have a real flight quote): {len(candidates)} destinations")
for flag in ["has_cost_of_living", "has_airbnb_data", "has_weather_data", "has_tourism_stats"]:
    if flag in candidates.columns:
        print(f"  of those, {flag:<22s}{int(candidates[flag].sum())}")

print("\nCandidate preview:")
print(candidates[["city", "country", "flight_price_avg_usd", "cost_of_living_composite",
                  "avg_airbnb_price_usd", "has_airbnb_data"]].sort_values("city").head(20).to_string(index=False))


COVERAGE: union list (this notebook) vs current Airbnb-based merge
metric                        Airbnb (current)    Union (proposed)
base size                                  102               4,999
flight destinations                         21                  49
cost of living                              64                4956
airbnb lodging                             102                 102
weather                                     95                 144
tourism                                    101                4775

Modeling candidates (have a real flight quote): 49 destinations
  of those, has_cost_of_living    44
  of those, has_airbnb_data       21
  of those, has_weather_data      47
  of those, has_tourism_stats     47

Candidate preview:
            city              country  flight_price_avg_usd  cost_of_living_composite  avg_airbnb_price_usd  has_airbnb_data
       Amsterdam          Netherlands                 450.5                  0.225301            293.2602